# Dense Retrieval Evaluation â€” BGE-M3 (Laws Corpus)

Evaluates BGE-M3 dense retrieval over the **full laws corpus** (~175K articles) using the corrected `CitationNormalizer` (251/251 val citations parse, including docket-style).

## What this notebook does
1. Embeds all laws with `BAAI/bge-m3` using the Kaggle GPU
2. Builds a FAISS `IndexFlatIP` (exact cosine similarity, cached to disk)
3. Evaluates on `val.csv` at k = 10, 15, 20, 25
4. Saves results to `results_dense_bge_m3.json`

## Kaggle setup required
- Attach **`omnilex-data`** dataset â†’ `laws_de.csv`, `val.csv`
- Attach **`omnilex-utils`** dataset â†’ the `omnilex` Python package
- Enable **GPU accelerator** (T4 Ã— 1)

In [ ]:
import os
print(os.listdir('/kaggle/input/datasets/hoangduongla'))

In [ ]:
import shutil, sys, os
from pathlib import Path

CODE_DST = Path('/kaggle/working/omnilex')

# Locate code: prefer mounted dataset, fallback to kagglehub download
MOUNTED = Path('/kaggle/input/datasets/hoangduongla/omnilex-retrieval-code')
if MOUNTED.exists() and any(MOUNTED.iterdir()):
    CODE_SRC = MOUNTED
    print(f'Using mounted dataset: {CODE_SRC}')
else:
    print('Code dataset not mounted â€” downloading via kagglehub...')
    import kagglehub
    CODE_SRC = Path(kagglehub.dataset_download('moeghri/omnilex-retrieval-code'))
    print(f'Downloaded to: {CODE_SRC}')

if CODE_DST.exists():
    shutil.rmtree(CODE_DST)
shutil.copytree(str(CODE_SRC), str(CODE_DST))

# Locate competition data
DATA_DIR = CODE_DST / 'data'
DATA_DIR.mkdir(exist_ok=True)
COMP_CANDIDATES = [
    Path('/kaggle/input/llm-agentic-legal-information-retrieval'),
    Path('/kaggle/input/competitions/llm-agentic-legal-information-retrieval'),
]
COMP_DIR = next((p for p in COMP_CANDIDATES if p.exists()), None)
if COMP_DIR is None:
    raise FileNotFoundError(
        'Competition data not found.\n'
        'Fix: click + Add Data in the sidebar, search for\n'
        '     llm-agentic-legal-information-retrieval, re-run this cell.'
    )
print(f'Competition data at: {COMP_DIR}')
for fname in ['laws_de.csv', 'court_considerations.csv', 'val.csv', 'test.csv']:
    src, dst = COMP_DIR / fname, DATA_DIR / fname
    if src.exists() and not dst.exists():
        dst.symlink_to(src)
        print(f'Linked {fname}')

# Both sys.path (this process) and PYTHONPATH (subprocesses) must point to src/
sys.path.insert(0, str(CODE_DST / 'src'))
os.environ['PYTHONPATH'] = str(CODE_DST / 'src')
os.chdir(str(CODE_DST))
print('Working dir:', os.getcwd())

# Output and index paths
import numpy as np
WORK_DIR     = Path('/kaggle/working')
LAWS_CSV     = DATA_DIR / 'laws_de.csv'
VAL_CSV      = DATA_DIR / 'val.csv'
INDEX_DIR    = WORK_DIR / 'bge_m3_laws_index'
RESULTS_JSON = WORK_DIR / 'results_dense_bge_m3.json'
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print(f'Laws CSV : {"found" if LAWS_CSV.exists() else "NOT FOUND"}')
print(f'Val CSV  : {"found" if VAL_CSV.exists() else "NOT FOUND"}')
print(f'Index    : {"cached" if (INDEX_DIR / "faiss.index").exists() else "(will build)"}')


In [ ]:
import subprocess, importlib

def _ok(m):
    try:
        importlib.import_module(m)
        return True
    except ImportError:
        return False

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True)
if not _ok('faiss'):
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu'], check=True)
        print('faiss-gpu installed.')
    except subprocess.CalledProcessError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'], check=True)
        print('faiss-cpu installed (faiss-gpu unavailable).')
else:
    print('faiss already available.')
print('Done.')


In [ ]:
import time, csv, json
import faiss
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

from omnilex.citations.normalizer import CitationNormalizer
from omnilex.evaluation.metrics import citation_f1

normalizer = CitationNormalizer()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device          : {device}')
if device == 'cuda':
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Sanity-check all three citation types
assert normalizer.canonicalize('Art. 221 Abs. 1 StPO') == 'Art. 221 Abs. 1 StPO'
assert normalizer.canonicalize('BGE 137 IV 122 E. 6.2') == 'BGE 137 IV 122 E. 6.2'
assert normalizer.canonicalize('1B_210/2023 E. 4.1')    == '1B_210/2023 E. 4.1'
print('Normalizer sanity check : OK')


In [ ]:
# ── Run configuration ─────────────────────────────────────────────────────────
RUN_DENSE = False   # dense retrieval confirmed as noise (Session 9); skip embedding

# Always load val data — needed by both dense and LLM pipelines
import csv as _csv_cfg
with open(str(VAL_CSV), encoding="utf-8") as _f:
    val_queries = list(_csv_cfg.DictReader(_f))

gold_sets = [
    set(normalizer.canonicalize_list(
        [c.strip() for c in q["gold_citations"].split(";") if c.strip()]
    ))
    for q in val_queries
]
gold_sets_by_qid = {q["query_id"]: gs for q, gs in zip(val_queries, gold_sets)}
total_gold = sum(len(g) for g in gold_sets)
print(f"RUN_DENSE = {RUN_DENSE}")
print(f"Val queries: {len(val_queries)}, total gold citations: {total_gold}")


## Load BGE-M3 model

In [ ]:
if RUN_DENSE:
    MODEL_NAME = "BAAI/bge-m3"
    BATCH_SIZE = 256 if device == "cuda" else 32

    print(f"Loading {MODEL_NAME} (downloads ~580 MB on first run)...")
    t0 = time.time()
    model = SentenceTransformer(MODEL_NAME, device=device)
    if device == "cuda":
        model.half()  # fp16: halves VRAM with negligible accuracy loss
    print(f"Model loaded in {time.time()-t0:.1f}s")
    print(f"Embedding dim   : {model.get_sentence_embedding_dimension()}")
    print(f"Batch size      : {BATCH_SIZE}")
else:
    print('RUN_DENSE=False — skipping dense cell')

## Build or load FAISS index

On first run (~20 min on T4 GPU): embeds all laws and saves the index. Subsequent runs load from cache in seconds.

In [ ]:
if RUN_DENSE:
    INDEX_FILE = INDEX_DIR / "faiss.index"
    META_FILE  = INDEX_DIR / "metadata.json"


    def build_and_save(laws_csv: Path, index_dir: Path):
        index_dir.mkdir(parents=True, exist_ok=True)

        # Load all laws
        records = []
        with open(laws_csv, encoding="utf-8") as f:
            for row in csv.DictReader(f):
                records.append({
                    "citation_raw": row["citation"],
                    "text": row.get("text", ""),
                })
        print(f"Loaded {len(records):,} laws")

        # Compose text: citation string prepended to passage text
        texts = [f"{r['citation_raw']} {r['text']}" for r in records]

        # Embed in batches with progress
        print(f"Embedding {len(texts):,} documents (batch={BATCH_SIZE})...")
        t0 = time.time()
        all_vecs = []
        for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Batches"):
            batch = texts[i : i + BATCH_SIZE]
            vecs = model.encode(
                batch,
                normalize_embeddings=True,
                show_progress_bar=False,
                convert_to_numpy=True,
            )
            all_vecs.append(vecs.astype(np.float32))
        embeddings = np.vstack(all_vecs)
        elapsed = time.time() - t0
        print(f"Embedding done in {elapsed/60:.1f} min  shape={embeddings.shape}")

        # FAISS IndexFlatIP: exact inner product (= cosine when vectors are unit-normed)
        dim = embeddings.shape[1]
        index = faiss.IndexFlatIP(dim)
        normed = embeddings.copy()
        faiss.normalize_L2(normed)
        index.add(normed)

        faiss.write_index(index, str(INDEX_FILE))
        with open(META_FILE, "w", encoding="utf-8") as f:
            json.dump(records, f, ensure_ascii=False)
        sz_mb = INDEX_FILE.stat().st_size / 1e6
        print(f"Saved to {INDEX_FILE}  ({sz_mb:.0f} MB)")
        return index, records


    if INDEX_FILE.exists() and META_FILE.exists():
        print("Loading cached index...")
        t0 = time.time()
        faiss_index = faiss.read_index(str(INDEX_FILE))
        with open(META_FILE, encoding="utf-8") as f:
            records = json.load(f)
        print(f"Loaded {faiss_index.ntotal:,} vectors in {time.time()-t0:.1f}s")
    else:
        faiss_index, records = build_and_save(LAWS_CSV, INDEX_DIR)

    print(f"\nIndex ready: {faiss_index.ntotal:,} laws")
else:
    print('RUN_DENSE=False — skipping dense cell')

## Evaluate on val.csv

In [ ]:
if RUN_DENSE:
    K_VALUES = [10, 15, 20, 25]

    with open(VAL_CSV, encoding="utf-8") as f:
        val_queries = list(csv.DictReader(f))
    print(f"Val queries: {len(val_queries)}")

    print("Embedding val queries...")
    query_vecs = model.encode(
        [q["query"] for q in val_queries],
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    ).astype(np.float32)
    faiss.normalize_L2(query_vecs)

    # Parse gold sets once using corrected normalizer
    gold_sets = [
        set(normalizer.canonicalize_list(
            [c.strip() for c in q["gold_citations"].split(";") if c.strip()]
        ))
        for q in val_queries
    ]

    total_gold = sum(len(g) for g in gold_sets)
    print(f"Total gold citations (normalizer-corrected): {total_gold}")

    # Search at max k once, slice per k
    _, idx_mat = faiss_index.search(query_vecs, max(K_VALUES))

    results_by_k = {}
    per_query_results = {}

    print()
    print(f"{'k':>4}  {'Macro F1':>10}  {'Mean Prec':>10}  {'Mean Rec':>10}")
    print("-" * 50)

    for k in K_VALUES:
        f1s, precs, recs = [], [], []
        details = []

        for qi, q in enumerate(val_queries):
            predicted = set(normalizer.canonicalize_list(
                [records[idx]["citation_raw"] for idx in idx_mat[qi, :k] if idx >= 0]
            ))
            gold = gold_sets[qi]
            s = citation_f1(list(predicted), list(gold))
            f1s.append(s["f1"])
            precs.append(s["precision"])
            recs.append(s["recall"])
            details.append({
                "query_id": q["query_id"],
                "gold_count": len(gold),
                "predicted_count": len(predicted),
                "tp": len(predicted & gold),
                "precision": round(s["precision"], 4),
                "recall": round(s["recall"], 4),
                "f1": round(s["f1"], 4),
            })

        macro = sum(f1s) / len(f1s)
        results_by_k[str(k)] = round(macro, 4)
        per_query_results[str(k)] = details
        print(f"{k:>4}  {macro:>10.4f}  {sum(precs)/len(precs):>10.4f}  {sum(recs)/len(recs):>10.4f}")

    print("-" * 50)
    best_k = max(results_by_k, key=results_by_k.get)
    print(f"Best k={best_k}  Macro F1={results_by_k[best_k]:.4f}")
else:
    print('RUN_DENSE=False — skipping dense cell')

In [ ]:
if RUN_DENSE:
    # Per-query breakdown at k=25
    k_show = str(25)
    print(f"Per-query results at k={k_show}:")
    print(f"{'Query':>10}  {'Gold':>5}  {'Pred':>5}  {'TP':>4}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}")
    print("-" * 58)
    for d in per_query_results[k_show]:
        print(f"{d['query_id']:>10}  {d['gold_count']:>5}  {d['predicted_count']:>5}  "
              f"{d['tp']:>4}  {d['precision']:>6.3f}  {d['recall']:>6.3f}  {d['f1']:>6.3f}")
else:
    print('RUN_DENSE=False — skipping dense cell')

## SUMMARY

In [ ]:
if RUN_DENSE:
    print("=" * 50)
    print("SUMMARY")
    print("=" * 50)
    print(f"Model          : {MODEL_NAME}")
    print(f"Corpus         : {faiss_index.ntotal:,} laws")
    print(f"Gold citations : {total_gold} (docket-aware normalizer)")
    print()
    print("Macro F1 by k:")
    for k, f1 in results_by_k.items():
        bar = "#" * int(f1 * 40)
        print(f"  k={k:>2}: {f1:.4f}  {bar}")
    print()
    print("Reference points:")
    print("  Oracle F1 @ k=25 : 0.7788  (ceiling)")
    print("  Anchor-only F1   : 0.0237  (floor)")
    print("  Leaderboard top  : 0.3590")
else:
    print('RUN_DENSE=False — skipping dense cell')

# SAVE RESULT

In [ ]:
if RUN_DENSE:
    import datetime

    results = {
        "model": MODEL_NAME,
        "corpus": "laws_de.csv",
        "corpus_size": faiss_index.ntotal,
        "normalizer_version": "with_docket_support",
        "val_queries": len(val_queries),
        "val_total_gold_citations": total_gold,
        "timestamp": datetime.datetime.now().isoformat(),
        "macro_f1_by_k": results_by_k,
        "per_query_at_k25": per_query_results.get("25", []),
    }

    with open(RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to {RESULTS_JSON}")
    print("\nmacro_f1_by_k:")
    print(json.dumps(results_by_k, indent=2))
else:
    print('RUN_DENSE=False — skipping dense cell')

# Anchor Extraction

In [ ]:
# ── Anchor Extraction ──────────────────────────────────────────────────────────
# Extract citations explicitly mentioned in each query (Art./BGE/docket patterns).
# These are free-precision signals: if the query names a citation, it is almost
# certainly relevant — regardless of whether it appears in the laws corpus.


!pip install -q rank-bm25
from omnilex.retrieval.anchor_extractor import extract_citation_anchors

anchor_lists = [extract_citation_anchors(q["query"]) for q in val_queries]

print("Anchor extraction per query:")
for q, anchors in zip(val_queries, anchor_lists):
    preview = anchors[:3]
    more = f"  (+{len(anchors)-3} more)" if len(anchors) > 3 else ""
    print(f"  {q['query_id']}: {len(anchors):>2} anchor(s)  {preview}{more}")

total_anchors = sum(len(a) for a in anchor_lists)
queries_with_anchors = sum(1 for a in anchor_lists if a)
print(f"\nTotal anchors: {total_anchors}  |  Queries with anchors: {queries_with_anchors}/{len(val_queries)}")

In [ ]:
# Map query_id -> anchor list (used by Task 2+ evaluation)
anchors_by_qid = {q['query_id']: al for q, al in zip(val_queries, anchor_lists)}

# Hybrid Evaluation

In [ ]:
if RUN_DENSE:
    # ── Hybrid Evaluation: Anchor + Dense via RRF ──────────────────────────────────
    # Reciprocal Rank Fusion (Cormack et al. 2009):
    #   score(c) = Σ  1 / (RRF_K + rank_in_channel)
    # k=60 is the standard constant that dampens rank-1 advantage.
    #
    # Two channels:
    #   1. Anchor  — citations extracted directly from query text (all citation types)
    #   2. Dense   — FAISS cosine similarity results from laws corpus
    #
    # Anchors cover court citations (BGE/docket) that are absent from laws_de.csv.
    # Dense covers laws the query doesn't name explicitly.

    RRF_K = 60
    DENSE_FETCH = 100  # fetch more candidates than k so RRF has room to re-rank

    def rrf_fuse(anchor_citations: list[str], dense_citations: list[str]) -> list[str]:
        scores: dict[str, float] = {}
        for rank, cit in enumerate(anchor_citations, start=1):
            scores[cit] = scores.get(cit, 0.0) + 1.0 / (RRF_K + rank)
        for rank, cit in enumerate(dense_citations, start=1):
            scores[cit] = scores.get(cit, 0.0) + 1.0 / (RRF_K + rank)
        return sorted(scores, key=scores.__getitem__, reverse=True)


    # Re-fetch dense candidates at DENSE_FETCH for RRF headroom
    _, idx_mat_large = faiss_index.search(query_vecs, DENSE_FETCH)

    dense_per_query = [
        normalizer.canonicalize_list([
            records[idx]["citation_raw"]
            for idx in idx_mat_large[qi]
            if idx >= 0
        ])
        for qi in range(len(val_queries))
    ]

    results_hybrid = {}
    per_query_hybrid = {}

    print(f"{'k':>4}  {'Hybrid F1':>10}  {'Dense F1':>10}  {'Delta':>8}")
    print("-" * 45)

    for k in K_VALUES:
        f1s = []
        details = []

        for qi, q in enumerate(val_queries):
            fused = rrf_fuse(anchor_lists[qi], dense_per_query[qi])[:k]
            predicted = set(fused)
            gold = gold_sets[qi]
            s = citation_f1(list(predicted), list(gold))
            f1s.append(s["f1"])
            details.append({
                "query_id":        q["query_id"],
                "gold_count":      len(gold),
                "anchors":         len(anchor_lists[qi]),
                "tp":              len(predicted & gold),
                "precision":       round(s["precision"], 4),
                "recall":          round(s["recall"], 4),
                "f1":              round(s["f1"], 4),
            })

        macro = round(sum(f1s) / len(f1s), 4)
        results_hybrid[str(k)] = macro
        per_query_hybrid[str(k)] = details

        dense_f1 = results_by_k[str(k)]
        delta = macro - dense_f1
        sign = "+" if delta >= 0 else ""
        print(f"{k:>4}  {macro:>10.4f}  {dense_f1:>10.4f}  {sign}{delta:>7.4f}")

    print("-" * 45)
    best_k = max(results_hybrid, key=results_hybrid.get)
    print(f"Best k={best_k}  Hybrid F1={results_hybrid[best_k]:.4f}")

    # Per-query breakdown at best k
    print(f"\nPer-query breakdown at k={best_k}:")
    print(f"{'Query':>10}  {'Gold':>5}  {'Anch':>5}  {'TP':>4}  {'F1':>6}")
    print("-" * 40)
    for d in per_query_hybrid[best_k]:
        print(f"{d['query_id']:>10}  {d['gold_count']:>5}  {d['anchors']:>5}  {d['tp']:>4}  {d['f1']:>6.3f}")
else:
    print('RUN_DENSE=False — skipping dense cell')

# Court Corpus Embedding

In [ ]:
if RUN_DENSE:
    # ── Courts Corpus Embedding ────────────────────────────────────────────────────
    # 2.4M court decisions — covers BGE and docket-style gold citations
    # Estimated runtime: 4–5 hours on T4 GPU

    import csv, time
    import faiss
    import numpy as np

    COURTS_CSV   = DATA_DIR / 'court_considerations.csv'
    COURTS_INDEX = WORK_DIR / 'bge_m3_courts_index'
    COURTS_INDEX.mkdir(exist_ok=True)

    COURTS_FAISS = COURTS_INDEX / 'faiss.index'
    COURTS_META  = COURTS_INDEX / 'metadata.json'

    if COURTS_FAISS.exists() and COURTS_META.exists():
        print("Loading cached courts index...")
        courts_index = faiss.read_index(str(COURTS_FAISS))
        with open(COURTS_META, encoding='utf-8') as f:
            courts_records = json.load(f)
        print(f"Loaded {courts_index.ntotal:,} court vectors")
    else:
        print("Loading courts CSV...")
        courts_records, texts = [], []
        with open(COURTS_CSV, encoding='utf-8') as f:
            for row in csv.DictReader(f):
                courts_records.append({"citation_raw": row["citation"]})
                parts = [p for p in [row.get("citation",""), row.get("text","")] if p]
                texts.append(" ".join(parts)[:512])

        print(f"Loaded {len(texts):,} court records — starting embedding...")
        t0 = time.time()
        courts_index = faiss.IndexFlatIP(1024)
        chunk_vecs = []

        for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Courts"):
            batch = texts[i:i+BATCH_SIZE]
            vecs = model.encode(batch, normalize_embeddings=True,
                                show_progress_bar=False, convert_to_numpy=True)
            chunk_vecs.append(vecs.astype(np.float32))
            if len(chunk_vecs) * BATCH_SIZE >= 5000 or (i + BATCH_SIZE) >= len(texts):
                block = np.vstack(chunk_vecs)
                faiss.normalize_L2(block)
                courts_index.add(block)
                chunk_vecs = []

        faiss.write_index(courts_index, str(COURTS_FAISS))
        with open(COURTS_META, 'w', encoding='utf-8') as f:
            json.dump(courts_records, f, ensure_ascii=False)
        print(f"Done in {(time.time()-t0)/3600:.1f}h — {courts_index.ntotal:,} vectors")
else:
    print('RUN_DENSE=False — skipping dense cell')

# 3 Way Hybrid Evaluation

In [ ]:
if RUN_DENSE:
    # ── 3-Way Hybrid Evaluation: Anchor + Laws + Courts ────────────────────────────
    # Now that courts index is built, add it as a third RRF channel.
    # Courts cover the 102/251 BGE/docket gold citations missing from laws corpus.

    DENSE_FETCH = 100

    def rrf_fuse_3way(anchor_cits, laws_cits, courts_cits):
        scores = {}
        for rank, cit in enumerate(anchor_cits, start=1):
            scores[cit] = scores.get(cit, 0.0) + 1.0 / (RRF_K + rank)
        for rank, cit in enumerate(laws_cits, start=1):
            scores[cit] = scores.get(cit, 0.0) + 1.0 / (RRF_K + rank)
        for rank, cit in enumerate(courts_cits, start=1):
            scores[cit] = scores.get(cit, 0.0) + 1.0 / (RRF_K + rank)
        return sorted(scores, key=scores.__getitem__, reverse=True)

    # Search courts index
    _, courts_idx_mat = courts_index.search(query_vecs, DENSE_FETCH)

    courts_per_query = [
        normalizer.canonicalize_list([
            courts_records[idx]["citation_raw"]
            for idx in courts_idx_mat[qi]
            if idx >= 0
        ])
        for qi in range(len(val_queries))
    ]

    results_3way = {}
    per_query_3way = {}

    print(f"{'k':>4}  {'3-Way F1':>10}  {'2-Way F1':>10}  {'Delta':>8}")
    print("-" * 45)

    for k in K_VALUES:
        f1s, details = [], []
        for qi, q in enumerate(val_queries):
            fused = rrf_fuse_3way(
                anchor_lists[qi],
                dense_per_query[qi],
                courts_per_query[qi]
            )[:k]
            predicted = set(fused)
            gold = gold_sets[qi]
            s = citation_f1(list(predicted), list(gold))
            f1s.append(s["f1"])
            details.append({
                "query_id": q["query_id"], "gold_count": len(gold),
                "tp": len(predicted & gold),
                "precision": round(s["precision"], 4),
                "recall": round(s["recall"], 4),
                "f1": round(s["f1"], 4),
            })

        macro = round(sum(f1s) / len(f1s), 4)
        results_3way[str(k)] = macro
        per_query_3way[str(k)] = details
        delta = macro - results_hybrid.get(str(k), 0)
        print(f"{k:>4}  {macro:>10.4f}  {results_hybrid.get(str(k),0):>10.4f}  {'+' if delta>=0 else ''}{delta:>7.4f}")

    print("-" * 45)
    best_k = max(results_3way, key=results_3way.get)
    print(f"Best k={best_k}  3-Way Hybrid F1={results_3way[best_k]:.4f}")

    print(f"\nPer-query at k={best_k}:")
    print(f"{'Query':>10}  {'Gold':>5}  {'TP':>4}  {'F1':>6}")
    print("-" * 35)
    for d in per_query_3way[best_k]:
        print(f"{d['query_id']:>10}  {d['gold_count']:>5}  {d['tp']:>4}  {d['f1']:>6.3f}")
else:
    print('RUN_DENSE=False — skipping dense cell')

# LLM Meta Filtering

In [ ]:
# ── Step 1: LLM Metadata Filtering ────────────────────────────────────────────
# Use KISSKI LLM to predict relevant Swiss law codes from the English query.
# This filters the laws corpus (~176K) down to relevant articles before search,
# cutting noise and improving precision dramatically.

!pip install -q openai

import os, json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
kisski_key = secrets.get_secret("KISSKI-API")

llm = OpenAI(
    api_key=kisski_key,
    base_url="https://chat-ai.academiccloud.de/v1"
)

# Top Swiss law code abbreviations covering ~95% of citation volume
SWISS_LAW_CODES = [
    "ZGB", "OR", "StGB", "StPO", "ZPO", "BGG", "BV",
    "SchKG", "IPRG", "AHVG", "IVG", "AVIG", "KVG", "UVG",
    "DBG", "MWSTG", "ArG", "VGG", "VwVG", "ATSG",
    "AIG", "AsylG", "BetmG", "GwG", "FinmIG", "FINIG",
    "BöB", "RPG", "USG", "SVG", "LMG", "MSchG", "URG"
]

PREDICT_PROMPT = """You are a Swiss law expert. Given an English legal query, predict which Swiss federal law codes are most likely to be cited.

Return ONLY a JSON array of law code abbreviations from this list:
{codes}

Rules:
- Return 2-6 codes maximum
- Only include codes clearly relevant to the legal issues in the query
- Return JSON array only, no explanation

Query: {query}"""

def predict_law_codes(query: str) -> list[str]:
    resp = llm.chat.completions.create(
        model="qwen3-30b-a3b-instruct-2507",
        messages=[{"role": "user", "content": PREDICT_PROMPT.format(
            codes=", ".join(SWISS_LAW_CODES),
            query=query[:1000]
        )}],
        max_tokens=100,
        temperature=0.0,
    )
    text = resp.choices[0].message.content.strip()
    # Strip thinking tags if present
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()
    try:
        codes = json.loads(text)
        return [c for c in codes if c in SWISS_LAW_CODES]
    except Exception:
        # Fallback: extract any known codes mentioned in response
        return [c for c in SWISS_LAW_CODES if c in text]

# Predict codes for all val queries
print("Predicting law codes per query...")
predicted_codes = {}
for q in val_queries:
    codes = predict_law_codes(q["query"])
    predicted_codes[q["query_id"]] = codes
    print(f"  {q['query_id']}: {codes}")
client = llm  # alias for Task 2+ cells


In [ ]:
if RUN_DENSE:
    # ── Step 2: Filtered Dense Retrieval ──────────────────────────────────────────
    # For each query, filter laws to predicted codes, search filtered subset,
    # then RRF with anchor channel. Falls back to full corpus if no codes predicted.

    def search_filtered(query_vec: np.ndarray, law_codes: list[str],
                        all_records: list[dict], faiss_idx,
                        top_k: int = 100) -> list[str]:
        if not law_codes:
            # No prediction — fall back to full corpus search
            _, idxs = faiss_idx.search(query_vec.reshape(1, -1), top_k)
            return normalizer.canonicalize_list([
                all_records[i]["citation_raw"] for i in idxs[0] if i >= 0
            ])

        # Filter record indices matching any predicted code
        pattern = "|".join(law_codes)
        import re
        filtered_idxs = [
            i for i, r in enumerate(all_records)
            if re.search(r'\b(' + pattern + r')\b', r["citation_raw"])
        ]

        if len(filtered_idxs) < 10:
            # Too few — fall back to full corpus
            _, idxs = faiss_idx.search(query_vec.reshape(1, -1), top_k)
            return normalizer.canonicalize_list([
                all_records[i]["citation_raw"] for i in idxs[0] if i >= 0
            ])

        # Search only within filtered subset
        filtered_vecs = np.vstack([
            faiss.downcast_index(faiss_idx).reconstruct(i)
            for i in filtered_idxs
        ]).astype(np.float32)
        sub_index = faiss.IndexFlatIP(filtered_vecs.shape[1])
        faiss.normalize_L2(filtered_vecs)
        sub_index.add(filtered_vecs)

        qv = query_vec.reshape(1, -1).copy()
        faiss.normalize_L2(qv)
        k = min(top_k, len(filtered_idxs))
        _, sub_idxs = sub_index.search(qv, k)

        return normalizer.canonicalize_list([
            all_records[filtered_idxs[i]]["citation_raw"]
            for i in sub_idxs[0] if i >= 0
        ])


    # Evaluate: anchor + filtered dense via RRF
    results_filtered = {}
    per_query_filtered = {}

    print(f"{'k':>4}  {'Filtered F1':>12}  {'2-Way F1':>10}  {'Delta':>8}")
    print("-" * 48)

    for k in K_VALUES:
        f1s, details = [], []
        for qi, q in enumerate(val_queries):
            codes = predicted_codes[q["query_id"]]
            dense_cits = search_filtered(query_vecs[qi], codes, records, faiss_index)
            fused = rrf_fuse(anchor_lists[qi], dense_cits)[:k]
            predicted = set(fused)
            gold = gold_sets[qi]
            s = citation_f1(list(predicted), list(gold))
            f1s.append(s["f1"])
            details.append({
                "query_id": q["query_id"], "predicted_codes": codes,
                "gold_count": len(gold), "tp": len(predicted & gold),
                "f1": round(s["f1"], 4),
            })

        macro = round(sum(f1s) / len(f1s), 4)
        results_filtered[str(k)] = macro
        per_query_filtered[str(k)] = details
        delta = macro - results_hybrid.get(str(k), 0)
        print(f"{k:>4}  {macro:>12.4f}  {results_hybrid.get(str(k),0):>10.4f}  {'+' if delta>=0 else ''}{delta:>7.4f}")

    print("-" * 48)
    best_k = max(results_filtered, key=results_filtered.get)
    print(f"Best k={best_k}  Filtered F1={results_filtered[best_k]:.4f}")

    print(f"\nPer-query at k={best_k}:")
    print(f"{'Query':>10}  {'Codes':>25}  {'TP':>4}  {'F1':>6}")
    print("-" * 55)
    for d in per_query_filtered[best_k]:
        codes_str = str(d['predicted_codes'])[:24]
        print(f"{d['query_id']:>10}  {codes_str:>25}  {d['tp']:>4}  {d['f1']:>6.3f}")
else:
    print('RUN_DENSE=False — skipping dense cell')

# BUILD CITATION BRIDGE

In [ ]:
# ── Citation Bridge Builder ────────────────────────────────────────────────────
# For each court decision, extract which law articles it cites.
# Output: {normalized_art_citation: [list of court citation strings]}
# Built once and cached. Takes ~25-35 min on 2.4M rows.

import re, json, csv
from collections import defaultdict
from pathlib import Path

BRIDGE_FILE = WORK_DIR / "citation_bridge.json"

# Reuse Art. pattern from anchor_extractor
_ART_RE = re.compile(
    r"(?:Art\.?|Artikel)\s*\d+[a-z]?"
    r"(?:\s+(?:Abs\.?|Absatz|al\.?|cpv\.?)\s*\d+[a-z]?)?"
    r"(?:\s+(?:lit\.?|Ziff\.?|Nr\.?)\s*[a-z0-9]+)?"
    r"\s+([A-Z][A-Za-z]{1,10})",
    re.UNICODE,
)

def extract_art_citations(text: str) -> list[str]:
    """Extract and normalize Art. citations from court decision text."""
    results = []
    for m in _ART_RE.finditer(text):
        raw = m.group(0).strip()
        canon = normalizer.canonicalize(raw)
        if canon:
            results.append(canon)
    return list(set(results))

if BRIDGE_FILE.exists():
    print("Loading cached citation bridge...")
    with open(BRIDGE_FILE, encoding="utf-8") as f:
        citation_bridge = json.load(f)
    print(f"Bridge loaded: {len(citation_bridge):,} law articles mapped")
else:
    print("Building citation bridge from court_considerations.csv...")
    print("(~25-35 min on 2.4M rows)")

    bridge = defaultdict(list)
    MAX_COURTS_PER_ART = 200  # cap to keep memory manageable

    t0 = time.time()
    row_count = 0

    with open(COURTS_CSV, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            court_cit = row.get("citation", "").strip()
            text = row.get("text", "")
            if not court_cit or not text:
                continue

            for art_cit in extract_art_citations(text):
                if len(bridge[art_cit]) < MAX_COURTS_PER_ART:
                    bridge[art_cit].append(court_cit)

            row_count += 1
            if row_count % 200_000 == 0:
                elapsed = time.time() - t0
                pct = row_count / 2_476_315 * 100
                eta = elapsed / max(row_count, 1) * (2_476_315 - row_count)
                print(f"  {row_count:>8,} rows ({pct:.1f}%)  elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m")

    citation_bridge = dict(bridge)

    with open(BRIDGE_FILE, "w", encoding="utf-8") as f:
        json.dump(citation_bridge, f, ensure_ascii=False)

    elapsed = time.time() - t0
    print(f"\nDone in {elapsed/60:.1f} min")
    print(f"Bridge: {len(citation_bridge):,} unique law articles")
    print(f"Sample: {list(citation_bridge.items())[:3]}")

In [ ]:
if RUN_DENSE:
    # ── Bridge-Enhanced Retrieval ──────────────────────────────────────────────────
    # For each retrieved law article, look up associated court decisions via bridge.
    # Three RRF channels: anchor + filtered_laws + bridge_courts

    def get_bridge_courts(law_citations: list[str], bridge: dict,
                          top_n: int = 50) -> list[str]:
        """Collect court citations linked to given law articles via the bridge."""
        seen, result = set(), []
        for art in law_citations:
            for court_cit in bridge.get(art, [])[:top_n]:
                canon = normalizer.canonicalize(court_cit)
                if canon and canon not in seen:
                    seen.add(canon)
                    result.append(canon)
        return result

    results_bridge = {}
    per_query_bridge = {}

    print(f"{'k':>4}  {'Bridge F1':>10}  {'Filtered F1':>12}  {'Delta':>8}")
    print("-" * 48)

    for k in K_VALUES:
        f1s, details = [], []
        for qi, q in enumerate(val_queries):
            codes = predicted_codes[q["query_id"]]
            # Filtered dense search for laws
            dense_laws = search_filtered(query_vecs[qi], codes, records, faiss_index, top_k=100)
            # Bridge: expand to court citations
            bridge_courts = get_bridge_courts(dense_laws[:5], citation_bridge, top_n=5)
            # 3-channel RRF: anchor + laws + bridge courts
            fused = rrf_fuse_3way(anchor_lists[qi], dense_laws, bridge_courts)[:k]
            predicted = set(fused)
            gold = gold_sets[qi]
            s = citation_f1(list(predicted), list(gold))
            f1s.append(s["f1"])
            details.append({
                "query_id": q["query_id"], "gold_count": len(gold),
                "bridge_courts": len(bridge_courts),
                "tp": len(predicted & gold), "f1": round(s["f1"], 4),
            })

        macro = round(sum(f1s) / len(f1s), 4)
        results_bridge[str(k)] = macro
        per_query_bridge[str(k)] = details
        delta = macro - results_filtered.get(str(k), 0)
        print(f"{k:>4}  {macro:>10.4f}  {results_filtered.get(str(k),0):>12.4f}  {'+' if delta>=0 else ''}{delta:>7.4f}")

    print("-" * 48)
    best_k = max(results_bridge, key=results_bridge.get)
    print(f"Best k={best_k}  Bridge F1={results_bridge[best_k]:.4f}")

    print(f"\nPer-query at k={best_k}:")
    print(f"{'Query':>10}  {'Gold':>5}  {'Bridge':>7}  {'TP':>4}  {'F1':>6}")
    print("-" * 42)
    for d in per_query_bridge[best_k]:
        print(f"{d['query_id']:>10}  {d['gold_count']:>5}  {d['bridge_courts']:>7}  {d['tp']:>4}  {d['f1']:>6.3f}")
else:
    print('RUN_DENSE=False — skipping dense cell')

# HyDE

In [ ]:
if RUN_DENSE:
    # ── HyDE: Hypothetical Document Embeddings ────────────────────────────────────
    # Instead of embedding the English query directly, ask the LLM to generate
    # a hypothetical German legal document that would answer the query.
    # This bridges the cross-lingual gap: German hypothesis → German corpus.

    def generate_hyde_doc(query: str) -> str:
        resp = llm.chat.completions.create(
            model="qwen3-30b-a3b-instruct-2507",
            messages=[{"role": "user", "content": f"""You are a Swiss law expert. Write a short hypothetical Swiss legal article excerpt in German (2-3 sentences) that would directly answer this legal question. Write ONLY the German legal text, no explanation.

    Question: {query[:800]}"""}],
            max_tokens=200,
            temperature=0.3,
        )
        text = resp.choices[0].message.content.strip()
        if "</think>" in text:
            text = text.split("</think>")[-1].strip()
        return text

    # Generate HyDE docs for all val queries
    print("Generating HyDE documents...")
    hyde_docs = {}
    for q in val_queries:
        doc = generate_hyde_doc(q["query"])
        hyde_docs[q["query_id"]] = doc
        print(f"\n  {q['query_id']}: {doc[:120]}...")

    # ── HyDE Evaluation: anchor + filtered HyDE dense via RRF ─────────────────────
    # Embed HyDE doc instead of raw English query, then filtered search + RRF.

    print("Embedding HyDE documents...")
    hyde_texts = [hyde_docs[q["query_id"]] for q in val_queries]
    hyde_vecs = model.encode(
        hyde_texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    ).astype(np.float32)
    faiss.normalize_L2(hyde_vecs)

    results_hyde = {}
    per_query_hyde = {}

    print(f"\n{'k':>4}  {'HyDE F1':>10}  {'Filtered F1':>12}  {'Delta':>8}")
    print("-" * 48)

    for k in K_VALUES:
        f1s, details = [], []
        for qi, q in enumerate(val_queries):
            codes = predicted_codes[q["query_id"]]
            dense_cits = search_filtered(hyde_vecs[qi], codes, records, faiss_index)
            fused = rrf_fuse(anchor_lists[qi], dense_cits)[:k]
            predicted = set(fused)
            gold = gold_sets[qi]
            s = citation_f1(list(predicted), list(gold))
            f1s.append(s["f1"])
            details.append({
                "query_id": q["query_id"], "gold_count": len(gold),
                "tp": len(predicted & gold), "f1": round(s["f1"], 4),
            })

        macro = round(sum(f1s) / len(f1s), 4)
        results_hyde[str(k)] = macro
        per_query_hyde[str(k)] = details
        delta = macro - results_filtered.get(str(k), 0)
        print(f"{k:>4}  {macro:>10.4f}  {results_filtered.get(str(k),0):>12.4f}  {'+' if delta>=0 else ''}{delta:>7.4f}")

    print("-" * 48)
    best_k = max(results_hyde, key=results_hyde.get)
    print(f"Best k={best_k}  HyDE F1={results_hyde[best_k]:.4f}")

    print(f"\nPer-query at k={best_k}:")
    print(f"{'Query':>10}  {'Gold':>5}  {'TP':>4}  {'F1':>6}")
    print("-" * 35)
    for d in per_query_hyde[best_k]:
        print(f"{d['query_id']:>10}  {d['gold_count']:>5}  {d['tp']:>4}  {d['f1']:>6.3f}")
else:
    print('RUN_DENSE=False — skipping dense cell')

# 3-Way RRF

In [ ]:
# ── 3-Way RRF: anchor + filtered raw query + filtered HyDE ───────────────────
# HyDE alone hurt, but as a complementary signal it might add diversity.

print(f"\n{'k':>4}  {'3-way RRF':>10}  {'Filtered (2-way)':>16}  {'Delta':>8}")
print("-" * 54)

results_3way = {}
for k in K_VALUES:
    f1s = []
    for qi, q in enumerate(val_queries):
        codes = predicted_codes[q["query_id"]]
        dense_raw  = search_filtered(query_vecs[qi], codes, records, faiss_index)
        dense_hyde = search_filtered(hyde_vecs[qi],  codes, records, faiss_index)
        anchors    = anchor_lists[qi]

        scores = {}
        for rank, c in enumerate(anchors,     start=1): scores[c] = scores.get(c, 0) + 1/(RRF_K+rank)
        for rank, c in enumerate(dense_raw,   start=1): scores[c] = scores.get(c, 0) + 1/(RRF_K+rank)
        for rank, c in enumerate(dense_hyde,  start=1): scores[c] = scores.get(c, 0) + 1/(RRF_K+rank)
        fused = sorted(scores, key=scores.__getitem__, reverse=True)[:k]

        gold = gold_sets[qi]
        f1s.append(citation_f1(fused, list(gold))["f1"])

    macro = round(sum(f1s)/len(f1s), 4)
    results_3way[str(k)] = macro
    delta = macro - results_filtered.get(str(k), 0)
    print(f"{k:>4}  {macro:>10.4f}  {results_filtered.get(str(k),0):>16.4f}  {'+' if delta>=0 else ''}{delta:>7.4f}")

print("-" * 54)
best_k = max(results_3way, key=results_3way.get)
print(f"Best k={best_k}  3-way RRF F1={results_3way[best_k]:.4f}")

In [ ]:
if RUN_DENSE:
    # ── Diagnostic: what does the top-25 for val_004 actually retrieve? ──────────
    diag_qi = next(i for i, q in enumerate(val_queries) if q["query_id"] == "val_004")
    diag_q  = val_queries[diag_qi]

    print(f"Query: {diag_q['query'][:200]}\n")
    print(f"Gold citations ({len(gold_sets[diag_qi])}):")
    for g in sorted(gold_sets[diag_qi]):
        print(f"  {g}")

    print(f"\nHyDE doc: {hyde_docs['val_004']}\n")
    print(f"Predicted codes: {predicted_codes['val_004']}\n")

    codes = predicted_codes["val_004"]

    dense_raw = search_filtered(query_vecs[diag_qi], codes, records, faiss_index)
    print("Top-25 filtered raw query retrieval:")
    for rank, c in enumerate(dense_raw[:25], start=1):
        hit = "✓" if c in gold_sets[diag_qi] else " "
        print(f"  {rank:>3} {hit} {c}")

    print()
    dense_hyde = search_filtered(hyde_vecs[diag_qi], codes, records, faiss_index)
    print("Top-25 filtered HyDE retrieval:")
    for rank, c in enumerate(dense_hyde[:25], start=1):
        hit = "✓" if c in gold_sets[diag_qi] else " "
        print(f"  {rank:>3} {hit} {c}")

    print(f"\nGold article citations — present in laws corpus?")
    all_law_cits = {records[i]["citation_raw"] for i in range(len(records))}
    for g in sorted(gold_sets[diag_qi]):
        present = g in all_law_cits
        print(f"  {'FOUND' if present else 'MISSING':7}  {g}")
else:
    print('RUN_DENSE=False — skipping dense cell')

In [ ]:
# Does 3-way RRF keep improving past k=25?
for k in [25, 30, 35, 40, 50]:
    f1s = []
    for qi, q in enumerate(val_queries):
        codes = predicted_codes[q["query_id"]]
        dense_raw  = search_filtered(query_vecs[qi], codes, records, faiss_index)
        dense_hyde = search_filtered(hyde_vecs[qi],  codes, records, faiss_index)
        scores = {}
        for rank, c in enumerate(anchor_lists[qi], start=1): scores[c] = scores.get(c,0) + 1/(RRF_K+rank)
        for rank, c in enumerate(dense_raw,        start=1): scores[c] = scores.get(c,0) + 1/(RRF_K+rank)
        for rank, c in enumerate(dense_hyde,       start=1): scores[c] = scores.get(c,0) + 1/(RRF_K+rank)
        fused = sorted(scores, key=scores.__getitem__, reverse=True)[:k]
        f1s.append(citation_f1(fused, list(gold_sets[qi]))["f1"])
    print(f"k={k:>3}  F1={sum(f1s)/len(f1s):.4f}")

In [ ]:
# ── LLM Article Prediction: Full Evaluation ──────────────────────────────────
import time

print("Generating LLM article predictions for all 10 val queries...")
llm_predicted = {}
for q in val_queries:
    preds = predict_articles_v2(q["query"], predicted_codes[q["query_id"]])
    # Also add parent citations (Art. 467 ZGB covers Art. 467 Abs. 1 ZGB)
    parents = set()
    for p in preds:
        parent = re.sub(r'\s+Abs\.\s*\d+', '', p).strip()
        if parent != p:
            parents.add(parent)
    llm_predicted[q["query_id"]] = list(dict.fromkeys(preds + list(parents)))
    print(f"  {q['query_id']}: {len(llm_predicted[q['query_id']])} articles")
    time.sleep(0.5)  # avoid rate limits

print("\n── Results: LLM prediction + anchor (union) ──")
print(f"{'Query':>10}  {'Gold':>5}  {'LLM_TP':>7}  {'Anc_TP':>7}  {'Union_TP':>9}  {'Union_F1':>9}")
print("-" * 58)

f1s_llm, f1s_union = [], []
for qi, q in enumerate(val_queries):
    gold = gold_sets[qi]
    preds_llm = set(llm_predicted[q["query_id"]])
    preds_anc = set(anchor_lists[qi])
    preds_union = preds_llm | preds_anc

    tp_llm   = len(preds_llm   & gold)
    tp_anc   = len(preds_anc   & gold)
    tp_union = len(preds_union & gold)

    s_llm   = citation_f1(list(preds_llm),   list(gold))
    s_union = citation_f1(list(preds_union), list(gold))
    f1s_llm.append(s_llm["f1"])
    f1s_union.append(s_union["f1"])

    print(f"{q['query_id']:>10}  {len(gold):>5}  {tp_llm:>7}  {tp_anc:>7}  {tp_union:>9}  {s_union['f1']:>9.3f}")

print("-" * 58)
print(f"{'Macro F1':>55}")
print(f"  LLM only:         {sum(f1s_llm)/len(f1s_llm):.4f}")
print(f"  LLM + anchor:     {sum(f1s_union)/len(f1s_union):.4f}")
print(f"  3-way RRF k=30:   0.0678  (previous best)")

In [ ]:
# ── Debug val_002: why did regex extract 0 articles? ─────────────────────────
val002_q = next(q for q in val_queries if q["query_id"] == "val_002")
print(f"Query: {val002_q['query'][:300]}\n")
print(f"Predicted codes: {predicted_codes['val_002']}\n")

# Re-run and capture raw LLM output before regex
resp = llm.chat.completions.create(
    model="qwen3-30b-a3b-instruct-2507",
    messages=[{"role": "user", "content": f"""You are a Swiss federal court law clerk. Analyze this case and predict the exact law articles cited in the court decision.

Step 1 — Identify the 4-5 specific legal issues this case raises.
Step 2 — For each issue, give the 2-3 most specific Swiss law articles that directly address it. Use exact article numbers with paragraph numbers (e.g. Art. 469 Abs. 1 ZGB, Art. 505 Abs. 1 ZGB).

Primary law codes to focus on: {', '.join(predicted_codes['val_002'])}
Also include Art. 100 Abs. 1 BGG if the case reaches the Federal Court.

Output format:
Issue: [name]
Articles: Art. X Abs. Y ZGB, Art. Z OR

Case:
{val002_q['query'][:1500]}"""}],
    max_tokens=600,
    temperature=0.1,
)
raw_text = resp.choices[0].message.content.strip()
if "</think>" in raw_text:
    raw_text = raw_text.split("</think>")[-1].strip()

print("Raw LLM output:")
print(raw_text)
print()

# Test regex on it
pattern = r'Art\.\s*\d+(?:\s+Abs\.\s*\d+)?(?:\s+(?:lit\.\s*\w+))?\s+(?:ZGB|OR|StGB|StPO|ZPO|BGG|BV|SchKG|AHVG|IVG|AVIG|KVG|UVG|DBG|MWSTG|ArG|VGG|VwVG|ATSG|AIG|AsylG|BetmG)'
found = re.findall(pattern, raw_text)
print(f"Regex found {len(found)} matches: {found}")

In [ ]:
# ── French→German law abbreviation mapping ───────────────────────────────────
FR_TO_DE = {
    "LAI": "IVG", "LACI": "AVIG", "LAMal": "KVG", "LPGA": "ATSG",
    "LTF": "BGG", "CO": "OR", "CC": "ZGB", "CPP": "StPO", "CPC": "ZPO",
    "LIFD": "DBG", "LEtr": "AIG", "LAsi": "AsylG", "LCR": "SVG",
    "LTVA": "MWSTG", "LCD": "UWG", "LP": "SchKG",
}

def predict_articles_v3(query: str, codes: list[str]) -> list[str]:
    codes_str = ", ".join(codes)
    resp = llm.chat.completions.create(
        model="qwen3-30b-a3b-instruct-2507",
        messages=[{"role": "user", "content": f"""You are a Swiss federal court law clerk. Analyze this case and predict the exact law articles cited in the court decision.

IMPORTANT: Always use GERMAN law abbreviations (IVG not LAI, AVIG not LACI, KVG not LAMal, ATSG not LPGA, BGG not LTF, OR not CO, ZGB not CC, StPO not CPP, ZPO not CPC).

Step 1 — Identify the 4-5 specific legal issues this case raises.
Step 2 — For each issue, give the 2-3 most specific Swiss law articles that directly address it. Use exact article numbers with paragraph numbers (e.g. Art. 8 Abs. 1 AVIG, Art. 17 Abs. 1 IVG).

Primary law codes to focus on: {codes_str}
Also include Art. 100 Abs. 1 BGG if the case reaches the Federal Court.

Output format:
Issue: [name]
Articles: Art. X Abs. Y IVG, Art. Z AVIG

Case:
{query[:1500]}"""}],
        max_tokens=600,
        temperature=0.1,
    )
    text = resp.choices[0].message.content.strip()
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()

    # Normalize French abbreviations to German
    for fr, de in FR_TO_DE.items():
        text = re.sub(rf'\b{fr}\b', de, text)

    pattern = r'Art\.\s*\d+(?:\s+Abs\.\s*\d+)?(?:\s+lit\.\s*\w+)?\s+(?:ZGB|OR|StGB|StPO|ZPO|BGG|BV|SchKG|AHVG|IVG|AVIG|KVG|UVG|DBG|MWSTG|ArG|VGG|VwVG|ATSG|AIG|AsylG|BetmG|SVG|UWG)'
    raw = re.findall(pattern, text)
    seen, deduped = set(), []
    for c in raw:
        norm = re.sub(r'\s+', ' ', c).strip()
        if norm not in seen:
            seen.add(norm)
            deduped.append(norm)
    # Add parent citations (strip Abs.)
    parents = []
    for p in deduped:
        parent = re.sub(r'\s+Abs\.\s*\d+', '', p).strip()
        if parent != p and parent not in seen:
            seen.add(parent)
            parents.append(parent)
    return deduped + parents

# Quick test on val_002
val002_q = next(q for q in val_queries if q["query_id"] == "val_002")
preds_002 = predict_articles_v3(val002_q["query"], predicted_codes["val_002"])
print(f"val_002 v3: {len(preds_002)} articles")
for i, a in enumerate(preds_002, 1):
    hit = "✓" if a in gold_sets[1] else " "
    print(f"  {i:>3} {hit} {a}")
gold_hits = [g for g in gold_sets[1] if g in set(preds_002)]
print(f"\nGold recall val_002: {len(gold_hits)}/{len(gold_sets[1])} = {len(gold_hits)/len(gold_sets[1]):.1%}")

In [ ]:
# ── Full evaluation: v3 LLM prediction + anchor + filtered dense (RRF) ───────
import time

print("Generating LLM v3 predictions for all 10 val queries...")
llm_predicted_v3 = {}
for q in val_queries:
    preds = predict_articles_v3(q["query"], predicted_codes[q["query_id"]])
    llm_predicted_v3[q["query_id"]] = preds
    print(f"  {q['query_id']}: {len(preds)} articles")
    time.sleep(0.5)

K_EVAL = 15

print(f"\n── LLM v3 + anchor + dense RRF (k={K_EVAL}) ──")
print(f"{'Query':>10}  {'Gold':>5}  {'LLM_TP':>7}  {'Combined_TP':>12}  {'F1':>7}")
print("-" * 52)

f1s_combined = []
for qi, q in enumerate(val_queries):
    gold = gold_sets[qi]
    codes = predicted_codes[q["query_id"]]

    # Channel 1: LLM predicted articles
    preds_llm = set(llm_predicted_v3[q["query_id"]])

    # Channel 2: anchor extraction
    preds_anc = set(anchor_lists[qi])

    # Channel 3: filtered dense retrieval
    dense_cits = search_filtered(query_vecs[qi], codes, records, faiss_index)

    # RRF over all 3 channels, then union with LLM predictions (LLM = exact match, high confidence)
    scores = {}
    for rank, c in enumerate(anchor_lists[qi], start=1): scores[c] = scores.get(c,0) + 1/(RRF_K+rank)
    for rank, c in enumerate(dense_cits,       start=1): scores[c] = scores.get(c,0) + 1/(RRF_K+rank)
    rrf_top = sorted(scores, key=scores.__getitem__, reverse=True)[:K_EVAL]

    # LLM predictions are high-confidence — take all of them + RRF top-k
    combined = list(preds_llm) + [c for c in rrf_top if c not in preds_llm]

    tp_llm  = len(preds_llm & gold)
    tp_comb = len(set(combined) & gold)
    s = citation_f1(combined, list(gold))
    f1s_combined.append(s["f1"])
    print(f"{q['query_id']:>10}  {len(gold):>5}  {tp_llm:>7}  {tp_comb:>12}  {s['f1']:>7.3f}")

print("-" * 52)
print(f"  Macro F1 (LLM v3 + anchor + dense): {sum(f1s_combined)/len(f1s_combined):.4f}")
print(f"  LLM + anchor only (v2):              0.0926")
print(f"  3-way RRF k=30 (previous best):      0.0678")

In [ ]:
# ── LLM prediction with 3-run ensemble per query ─────────────────────────────
def predict_articles_ensemble(query: str, codes: list[str], runs: int = 3) -> list[str]:
    seen, result = set(), []
    for _ in range(runs):
        preds = predict_articles_v3(query, codes)
        for p in preds:
            if p not in seen:
                seen.add(p)
                result.append(p)
        time.sleep(0.3)
    return result

print("Generating ensemble predictions (3 runs × 10 queries = 30 LLM calls)...")
llm_predicted_ens = {}
for q in val_queries:
    preds = predict_articles_ensemble(q["query"], predicted_codes[q["query_id"]], runs=3)
    llm_predicted_ens[q["query_id"]] = preds
    print(f"  {q['query_id']}: {len(preds)} articles")
    time.sleep(0.5)

# Evaluate LLM ensemble + anchor
print(f"\n── Ensemble LLM + anchor ──")
f1s = []
for qi, q in enumerate(val_queries):
    gold = gold_sets[qi]
    combined = set(llm_predicted_ens[q["query_id"]]) | set(anchor_lists[qi])
    s = citation_f1(list(combined), list(gold))
    f1s.append(s["f1"])
    tp = len(combined & gold)
    print(f"  {q['query_id']}  gold={len(gold)}  tp={tp}  f1={s['f1']:.3f}")
print(f"\n  Macro F1: {sum(f1s)/len(f1s):.4f}")
print(f"  Previous best: 0.0926")

In [ ]:
# ── predict_articles_v4: cap at 10, force precision over recall ───────────────
def predict_articles_v4(query: str, codes: list[str]) -> list[str]:
    codes_str = ", ".join(codes)
    resp = llm.chat.completions.create(
        model="qwen3-30b-a3b-instruct-2507",
        messages=[{"role": "user", "content": f"""You are a Swiss federal court law clerk. List the 8 to 10 most likely article citations for this case — only articles you are highly confident the court cites.

RULES:
- GERMAN abbreviations only: IVG not LAI, AVIG not LACI, BGG not LTF, OR not CO, ZGB not CC
- Include the paragraph number when you are confident: Art. 469 Abs. 1 ZGB
- Omit the paragraph if you are not sure which one: Art. 469 ZGB
- Do NOT list sequential articles (e.g. 490, 491, 492...) — pick only specific relevant ones
- Maximum 10 citations total
- Always include Art. 100 Abs. 1 BGG for Federal Court cases

Primary codes: {codes_str}

Output: one citation per line, nothing else.

Case:
{query[:1500]}"""}],
        max_tokens=300,
        temperature=0.0,
    )
    text = resp.choices[0].message.content.strip()
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()

    for fr, de in FR_TO_DE.items():
        text = re.sub(rf'\b{fr}\b', de, text)

    pattern = r'Art\.\s*\d+(?:\s+Abs\.\s*\d+)?(?:\s+lit\.\s*\w+)?\s+(?:ZGB|OR|StGB|StPO|ZPO|BGG|BV|SchKG|AHVG|IVG|AVIG|KVG|UVG|DBG|MWSTG|ArG|VGG|VwVG|ATSG|AIG|AsylG|BetmG|SVG|UWG)'
    raw = re.findall(pattern, text)
    seen, deduped = set(), []
    for c in raw:
        norm = re.sub(r'\s+', ' ', c).strip()
        if norm not in seen:
            seen.add(norm)
            deduped.append(norm)
    parents = []
    for p in deduped:
        parent = re.sub(r'\s+Abs\.\s*\d+', '', p).strip()
        if parent != p and parent not in seen:
            seen.add(parent)
            parents.append(parent)
    return deduped + parents

# Quick test on val_004 and val_007 (the problematic ones)
for qid, qi in [("val_004", 3), ("val_007", 6)]:
    test_q = next(q for q in val_queries if q["query_id"] == qid)
    preds = predict_articles_v4(test_q["query"], predicted_codes[qid])
    tp = len(set(preds) & gold_sets[qi])
    s = citation_f1(preds, list(gold_sets[qi]))
    print(f"{qid}: {len(preds)} articles, {tp} TP, F1={s['f1']:.3f}")
    for i, a in enumerate(preds, 1):
        hit = "✓" if a in gold_sets[qi] else " "
        print(f"  {i:>3} {hit} {a}")
    print()

# Task 2 — predict_articles_v4: deterministic, confidence-ordered

In [ ]:
import re as _re

LLM_MODEL = "qwen3-30b-a3b-instruct-2507"   # Session 9: outperformed 122b on this task
MAX_ARTICLES_REQUEST = 25                    # request many, truncate later (Task 3 calibrates)

V4_SYSTEM = (
    "You are a Swiss legal expert. Given a case description, determine which "
    "Swiss federal law articles a court would cite when deciding this case.\n\n"
    "Work in two steps:\n\n"
    "ISSUES:\n"
    "List the distinct legal issues in the case, one line each.\n\n"
    "CITATIONS:\n"
    "List the specific applicable articles, ONE PER LINE, format:\n"
    "Art. <number> [Abs. <number>] <GERMAN abbreviation>\n"
    "Example: Art. 221 Abs. 1 StPO\n\n"
    "Rules:\n"
    "- Use GERMAN law abbreviations only (ZGB, OR, StGB, StPO, ZPO, SchKG, BGG, BV, "
    "IVG, AVIG, UVG, KVG, ATSG, ...). Never French (CC, CO, CP, CPP, LAI, LACI) or Italian.\n"
    "- Order citations from MOST confident to LEAST confident.\n"
    "- List at most {n} citations. Quality over quantity.\n"
    "- After the citation list, output nothing else."
)


def predict_articles_v4(query: str, n: int = MAX_ARTICLES_REQUEST) -> list:
    """Deterministic article prediction. Returns canonical citations, confidence-ordered."""
    resp = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": V4_SYSTEM.format(n=n)},
            {"role": "user", "content": query},
        ],
        temperature=0.0,
        max_tokens=2000,
    )
    text = (resp.choices[0].message.content or "").strip()
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()
    if "CITATIONS" in text:
        text = text.split("CITATIONS", 1)[1]

    out, seen = [], set()
    for line in text.splitlines():
        line = line.strip(" -*\u2022:\t")
        if "Art." not in line:
            continue
        line = line[line.index("Art."):]
        for fr, de in FR_TO_DE.items():
            line = _re.sub(rf"\b{_re.escape(fr)}\b", de, line)
        canon = normalizer.canonicalize(line)
        if canon and canon not in seen:
            seen.add(canon)
            out.append(canon)
    return out


def expand_parents(citations: list) -> list:
    """Add Abs.-stripped parent form for every prediction that has Abs. (Session 9 fix)."""
    out, seen = [], set()
    for c in citations:
        for form in (c, _re.sub(r"\s+Abs\.\s+\d+", "", c)):
            if form not in seen:
                seen.add(form)
                out.append(form)
    return out


print("predict_articles_v4 and expand_parents defined")


In [ ]:
def macro_f1_for(preds_by_qid: dict):
    """preds_by_qid: query_id -> iterable of canonical citation strings."""
    rows, f1s = [], []
    for q in val_queries:
        qid = q["query_id"]
        gold = gold_sets_by_qid[qid]
        pred = set(preds_by_qid.get(qid, []))
        s = citation_f1(list(pred), list(gold))
        f1s.append(s["f1"])
        rows.append({
            "query_id": qid, "gold": len(gold), "pred": len(pred),
            "tp": len(pred & gold), "f1": round(s["f1"], 4),
        })
    return sum(f1s) / len(f1s), rows

print("macro_f1_for defined")


In [ ]:
# Determinism check: 3 runs on val_004 (stochastic in Session 9 at temp=0.1)
val_004 = next(q for q in val_queries if q["query_id"] == "val_004")
runs = [predict_articles_v4(val_004["query"]) for _ in range(3)]
print("Run lengths:", [len(r) for r in runs])
print("Identical across runs:", runs[0] == runs[1] == runs[2])
gold4 = gold_sets_by_qid["val_004"]
for i, r in enumerate(runs):
    tp = len(set(expand_parents(r)) & gold4)
    print(f"  run {i}: {len(r)} articles, {tp} TP")


In [ ]:
from tqdm.auto import tqdm as _tqdm_v4

v4_preds_ordered = {}
for q in _tqdm_v4(val_queries, desc="v4 predict"):
    v4_preds_ordered[q["query_id"]] = predict_articles_v4(q["query"])

v4_full = {
    qid: set(expand_parents(lst)) | set(anchors_by_qid.get(qid, []))
    for qid, lst in v4_preds_ordered.items()
}
macro, rows = macro_f1_for(v4_full)
print(f"v4 (untruncated, n=25) + anchor: Macro F1 = {macro:.4f}   [v2 baseline: 0.0926]")
for r in rows:
    print(r)


# Task 3 — Citation-count calibration: truncation sweep

In [ ]:
print(f"{'N':>4}  {'Macro F1':>9}  {'mean pred':>9}")
sweep = {}
for N in [5, 8, 10, 12, 15, 20, 25]:
    preds = {
        qid: set(expand_parents(lst[:N])) | set(anchors_by_qid.get(qid, []))
        for qid, lst in v4_preds_ordered.items()
    }
    macro, rows = macro_f1_for(preds)
    sweep[N] = macro
    mean_pred = sum(r["pred"] for r in rows) / len(rows)
    print(f"{N:>4}  {macro:>9.4f}  {mean_pred:>9.1f}")

BEST_N = max(sweep, key=sweep.get)
print(f"\nBest N = {BEST_N}, Macro F1 = {sweep[BEST_N]:.4f}  [previous best: 0.0926]")

v4_best = {
    qid: set(expand_parents(lst[:BEST_N])) | set(anchors_by_qid.get(qid, []))
    for qid, lst in v4_preds_ordered.items()
}


# Task 4 — val_007 diagnostic

In [ ]:
import re as _re_diag

q7 = next(q for q in val_queries if q["query_id"] == "val_007")
gold7 = sorted(gold_sets_by_qid["val_007"])
pred7 = v4_preds_ordered.get("val_007", [])

print("QUERY:\n", q7["query"][:600], "\n")
print(f"GOLD ({len(gold7)}):")
for g in gold7:
    print("  ", g)
print(f"\nLLM v4 PREDICTED ({len(pred7)}):")
for p in pred7:
    print("  ", p)


def _book(c):
    m = _re_diag.search(r"([A-Z\u00c4\u00d6\u00dc][A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc]+)$", c)
    return m.group(1) if m else "?"


def _artno(c):
    m = _re_diag.search(r"Art\.\s+(\d+)", c)
    return int(m.group(1)) if m else None


gold_books = {_book(g) for g in gold7 if g.startswith("Art.")}
pred_books = {_book(p) for p in pred7}
print(f"\nGold books     : {sorted(gold_books)}")
print(f"Predicted books: {sorted(pred_books)}")
print(f"Book overlap   : {sorted(gold_books & pred_books)}")
near = [
    (p, g)
    for p in pred7
    for g in gold7
    if g.startswith("Art.")
    and _book(p) == _book(g)
    and _artno(p) and _artno(g)
    and abs(_artno(p) - _artno(g)) <= 3
]
print(f"Near misses (same book, \u00b13 articles): {near}")
